In [1]:
!pip install tensorflow numpy -q


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("neisse/scrapped-lyrics-from-6-genres")

print("Path to dataset files:", path)

100%|██████████| 129M/129M [00:01<00:00, 124MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/neisse/scrapped-lyrics-from-6-genres/versions/3


In [3]:
import pandas as pd
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [4]:
import os

print(os.listdir(path))

['lyrics-data.csv', 'artists-data.csv']


In [5]:
df = pd.read_csv(
    os.path.join(path, "lyrics-data.csv")
)

In [6]:

df.head()


,ALink,SName,SLink,Lyric,language
0,/ivete-sangalo/,Arerê,/ivete-sangalo/arere.html,"Tudo o que eu quero nessa vida,\nToda vida, é\...",pt
1,/ivete-sangalo/,Se Eu Não Te Amasse Tanto Assim,/ivete-sangalo/se-eu-nao-te-amasse-tanto-assim...,Meu coração\nSem direção\nVoando só por voar\n...,pt
2,/ivete-sangalo/,Céu da Boca,/ivete-sangalo/chupa-toda.html,É de babaixá!\nÉ de balacubaca!\nÉ de babaixá!...,pt
3,/ivete-sangalo/,Quando A Chuva Passar,/ivete-sangalo/quando-a-chuva-passar.html,Quando a chuva passar\n\nPra quê falar\nSe voc...,pt
4,/ivete-sangalo/,Sorte Grande,/ivete-sangalo/sorte-grande.html,A minha sorte grande foi você cair do céu\nMin...,pt


In [7]:
df = df.dropna()

df = df.head(200)

text = " ".join(df["Lyric"].astype(str))

print("Total characters:", len(text))

Total characters: 171013


In [8]:
import re

text = text.lower()

text = re.sub(r'[^a-zA-Z\s]', '', text)

text = re.sub(r'\s+', ' ', text)

print(text[:500])

tudo o que eu quero nessa vida toda vida amar voc amar voc o seu amor como uma chama acesa queima de prazer de prazer eu j falei com deus que no vou te deixar vou te levar pra onde for qualquer lugar j fiz de tudo pra no te perder arer um lobby um hobby um love com voc arer um lobby um hobby um love com voc cai cai cai cai cai pra c hey hey hey tudotudo vai rolar meu corao sem direo voando s por voar sem saber onde chegar sonhando em te encontrar e as estrelas que hoje eu descobri no seu olhar a


In [9]:

tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts([text])

total_words = 5000

print("Vocabulary Size:", total_words)

Vocabulary Size: 5000


In [10]:
words = text.split()

sequences = []

window_size = 5

for i in range(window_size, len(words)):

    seq = words[i-window_size:i+1]

    sequences.append(seq)

print("Total sequences:", len(sequences))

Total sequences: 33167


In [11]:
print(sequences[0])
print(sequences[1])
print(sequences[2])
print(sequences[3])
print(sequences[4])

['tudo', 'o', 'que', 'eu', 'quero', 'nessa']
['o', 'que', 'eu', 'quero', 'nessa', 'vida']
['que', 'eu', 'quero', 'nessa', 'vida', 'toda']
['eu', 'quero', 'nessa', 'vida', 'toda', 'vida']
['quero', 'nessa', 'vida', 'toda', 'vida', 'amar']


In [12]:
input_sequences = []

for seq in sequences:

    encoded = tokenizer.texts_to_sequences([" ".join(seq)])[0]

    if len(encoded) == window_size + 1:
        input_sequences.append(encoded)

input_sequences = np.array(input_sequences)

print(input_sequences.shape)

(33167, 6)


In [13]:
print(input_sequences[0])
print(input_sequences[1])
print(input_sequences[2])
print(input_sequences[3])
print(input_sequences[4])

[ 38   6   1   5  23 203]
[  6   1   5  23 203  43]
[  1   5  23 203  43  62]
[  5  23 203  43  62  43]
[ 23 203  43  62  43  32]


In [14]:
X = input_sequences[:, :-1]

y = input_sequences[:, -1]

y = to_categorical(
    y,
    num_classes=total_words
)

print(X.shape)
print(y.shape)

(33167, 5)
(33167, 5000)


In [15]:
model = Sequential([

    Embedding(
        input_dim=total_words,
        output_dim=64,
        input_length=window_size
    ),

    SimpleRNN(128),

    Dense(
        total_words,
        activation='softmax'
    )

])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [16]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [17]:
history = model.fit(
    X,
    y,
    epochs=10,
    batch_size=128,
    validation_split=0.2
)


Epoch 1/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 12s 48ms/step - accuracy: 0.0253 - loss: 6.8970 - val_accuracy: 0.0256 - val_loss: 6.8565
Epoch 2/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 10s 48ms/step - accuracy: 0.0286 - loss: 6.4265 - val_accuracy: 0.0312 - val_loss: 6.9299
Epoch 3/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 8s 38ms/step - accuracy: 0.0419 - loss: 6.0986 - val_accuracy: 0.0354 - val_loss: 6.9258
Epoch 4/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - accuracy: 0.0688 - loss: 5.6758 - val_accuracy: 0.0424 - val_loss: 6.9350
Epoch 5/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - accuracy: 0.1055 - loss: 5.2635 - val_accuracy: 0.0437 - val_loss: 7.0376
Epoch 6/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.1456 - loss: 4.8793 - val_accuracy: 0.0424 - val_loss: 7.1218
Epoch 7/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 12s 59ms/step - accuracy: 0.1864 - loss: 4.5119 - val_accuracy: 0.0467 - val_loss: 7.1804
Epoch 8/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 19s 51ms/step - accuracy: 0.2363 - loss: 4.1705 - val_

In [18]:
def predict_next_word(seed_text):

    token_list = tokenizer.texts_to_sequences(
        [seed_text]
    )[0]

    token_list = token_list[-window_size:]

    token_list = pad_sequences(
        [token_list],
        maxlen=window_size
    )

    predicted = np.argmax(
        model.predict(token_list, verbose=0)
    )

    for word, index in tokenizer.word_index.items():

        if index == predicted:

            return word

    return ""

In [19]:
def generate_text(seed_text, n_words):

    for _ in range(n_words):

        next_word = predict_next_word(seed_text)

        seed_text += " " + next_word

    return seed_text

In [25]:

def generate_text(seed_text, n_words):

    for _ in range(n_words):

        next_word = predict_next_word(seed_text)

        seed_text += " " + next_word

    return seed_text


print(generate_text("i am the person", 5))

print(generate_text("hello from the", 5))

print(generate_text("i am love", 5))


i am the person way i hey i never
hello from the chica its human and ive
i am love to hey hey i i
